<table class="table table-bordered">
    <tr>
        <th style="text-align:center; width:25%"><img src='https://www.np.edu.sg/_next/image?url=https%3A%2F%2Fassets.app.optical.gov.sg%2Fnp%2Fproduction%2Fpublished%2Fcollections%2Fpages%2F17c64bb4-8632-49e9-af16-047f7cabe99a%2Fab16008f-a6a0-466e-9193-e16e947261b0.png&w=1080&q=75'"></th>
        <th style="text-align:center;"><hrfm>SCHOOL OF INFOCOMM TECHNOLOGY <br> Diploma in Data Science</hrfm><h2>Machine Learning Data: Assignment Coding</h2></th>
    </tr>
</table>


<span style="color:blue; font-size:1.2em;">
<strong>Note: This template provides suggested steps to guide your work. You may include additional steps where appropriate.</strong>
</span><br><br>


## Import the required packages

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Data preparation
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from feature_engine.outliers import Winsorizer
from sklearn.preprocessing import PowerTransformer

# Regression models
from sklearn.linear_model import LinearRegression
from sklearn import tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

# Model evaluation
from sklearn.metrics import mean_squared_error, mean_absolute_error, silhouette_score
from math import sqrt

# Clustering models
from sklearn.cluster import KMeans, AgglomerativeClustering
import scipy.cluster.hierarchy as shc

# Statistical model
import statsmodels.api as sm

## 1. Task 1: Supervised Learning Using Wine Quality Dataset

### 1.1 Load and Explore the data


In [ ]:
df = pd.read_csv('./data/winequality-white.csv')

In [ ]:
#Basic Checks

print("\nHead")
print(df.head())

print("\nShape")
print(df.shape)

print("\nInfo")
print(df.info())

print("\nDescribe")
print(df.describe())

print("\nNull values")
print(df.isnull().sum())

#The dataset contains numeric chemical features and the target variable `quality`. 
# There are no missing values, so no missing value treatment is required for this dataset.

In [ ]:
#Plot Historgram for Distribution of Vraiables
df.hist(bins=30, figsize=(14, 10))
plt.suptitle("Distribution of Variables")
plt.show()

#The histograms show that most features are not perfectly normally distributed. 
#Some variables such as residual sugar, chlorides, and free sulfur dioxide are right-skewed. 
#The `quality` values are mostly concentrated around 5 to 7, especially 6.

In [ ]:
#Plot Boxplot
for i in df.drop(columns='quality').columns:
    plt.figure(figsize=(5, 4))
    plt.boxplot(df[i])
    plt.title(f"Boxplot of {i}")
    plt.ylabel("Value")
    plt.show()

#The boxplots show possible outliers in several chemical features, especially residual sugar, chlorides, and sulfur dioxide variables. 



In [ ]:
#Plot Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

#The correlation heatmap shows that alcohol has the strongest positive correlation with wine quality. 
#Density, chlorides, volatile acidity, and total sulfur dioxide have negative relationships with quality. 
#Most other relationships are weak to moderate.

### 1.2 Data Preparation

In [ ]:
# Separate input features and target variable
X = df.drop(columns='quality')
y = df['quality']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Cap outliers using the IQR method
winsorizer = Winsorizer(
    capping_method='iqr',
    tail='both',
    fold=1.5,
    variables=X_train.columns.tolist()
)

winsorizer.fit(X_train)

X_train_capped = winsorizer.transform(X_train)
X_test_capped = winsorizer.transform(X_test)

#Outliers are capped using the IQR method. 
#The winsorizer is fitted only on the training data, then applied to both training and testing data to avoid data leakage.

In [ ]:


# Select only clearly skewed features
skewed_features = [
    'volatile acidity',
    'citric acid',
    'residual sugar',
    'chlorides',
    'free sulfur dioxide',
    'sulphates'
]

# Make copies
X_train_transformed = X_train_capped.copy()
X_test_transformed = X_test_capped.copy()

# Apply Yeo-Johnson transformation only to skewed features
power_transformer = PowerTransformer(
    method='yeo-johnson',
    standardize=False
)

power_transformer.fit(X_train_capped[skewed_features])

X_train_transformed[skewed_features] = power_transformer.transform(
    X_train_capped[skewed_features]
)

X_test_transformed[skewed_features] = power_transformer.transform(
    X_test_capped[skewed_features]
)

In [ ]:
# Standardise features
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_transformed)
X_test_scaled = scaler.transform(X_test_transformed)

#Standardisation is applied so that all features have a similar scale.
#This is important for distance-based models such as K-Nearest Neighbours.

In [ ]:
# Convert scaled arrays back to DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

#The scaled arrays are converted back into DataFrames to keep the original column names, making the data easier to inspect and use in later models.

### 1.3 Build the Machine Learning Models

In [ ]:
# Build Linear Regression model
baseline_linear = LinearRegression()
baseline_linear.fit(X_train_scaled, y_train)

y_fitted_linear = baseline_linear.predict(X_train_scaled)
y_pred_linear = baseline_linear.predict(X_test_scaled)

In [ ]:
# Build Decision Tree Regressor model
baseline_tree = DecisionTreeRegressor(max_depth=2, random_state=42)
baseline_tree.fit(X_train_capped, y_train)

y_fitted_tree = baseline_tree.predict(X_train_capped)
y_pred_tree = baseline_tree.predict(X_test_capped)

In [ ]:
# Build K-Nearest Neighbours Regressor model
baseline_knn = KNeighborsRegressor(n_neighbors=5)
baseline_knn.fit(X_train_scaled, y_train)

y_fitted_knn = baseline_knn.predict(X_train_scaled)
y_pred_knn = baseline_knn.predict(X_test_scaled)

### 1.4 Evaluate and Improve the Model(s) Performance

In [ ]:
#Baseline model evaluation
print("Linear Regression Model")
print(sqrt(mean_squared_error(y_train, y_fitted_linear)), '(Train RMSE)')
print(baseline_linear.score(X_train_scaled, y_train), '(Train R^2 Value)')

print(sqrt(mean_squared_error(y_test, y_pred_linear)), '(Test RMSE)')
print(baseline_linear.score(X_test_scaled, y_test), '(Test R^2 Value)')

print("\nTree Regressor Model")
print(sqrt(mean_squared_error(y_train, y_fitted_tree)), '(Train RMSE)')
print(baseline_tree.score(X_train_capped, y_train), '(Train R^2 Value)')

print(sqrt(mean_squared_error(y_test, y_pred_tree)), '(Test RMSE)')
print(baseline_tree.score(X_test_capped, y_test), '(Test R^2 Value)')

print("\nKNN Model")
print(sqrt(mean_squared_error(y_train, y_fitted_knn)), '(Train RMSE)')
print(baseline_knn.score(X_train_scaled, y_train), '(Train R^2 Value)')

print(sqrt(mean_squared_error(y_test, y_pred_knn)), '(Test RMSE)')
print(baseline_knn.score(X_test_scaled, y_test), '(Test R^2 Value)')



In [ ]:
#Tuning for Linear Regression
lm2 = sm.OLS(y_train, X_train_capped).fit()
lm2.summary()

#The OLS summary is used to inspect the statistical significance of each feature. 
#Features with high p-values may have weaker contribution to the Linear Regression model.
# Based on the OLS results, `citric acid` was removed and the Linear Regression model was rebuilt.

In [ ]:
#Linear Regression after feature selection

X_train_selected = X_train_scaled.drop(columns=['citric acid'])
X_test_selected = X_test_scaled.drop(columns=['citric acid'])

linear_model = LinearRegression()
linear_model.fit(X_train_selected, y_train)

y_fitted_linear_selected = linear_model.predict(X_train_selected)
y_pred_linear_selected = linear_model.predict(X_test_selected)

In [ ]:
#GridSearchCV for Decision Tree
tree_reg = tree.DecisionTreeRegressor(random_state=42)

param_grid = { 
    "min_samples_leaf": [1, 5, 10], 
    "min_samples_split": [2, 4, 10, 12, 16], 
    "max_depth": [2, 3, 4, 5, 6, 7]
}

gs = GridSearchCV(
    tree_reg,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=10,
    n_jobs=-1
)

gs = gs.fit(X_train_capped, y_train)

print(gs.best_score_)
print(gs.best_params_)

In [ ]:
#Tuned Decision Tree model
decision_tree = tree.DecisionTreeRegressor(
    max_depth=6,
    min_samples_leaf=10,
    min_samples_split=2,
    random_state=42
)

decision_tree.fit(X_train_capped, y_train)

y_fitted_decision_tree = decision_tree.predict(X_train_capped)
y_pred_decision_tree = decision_tree.predict(X_test_capped)

In [ ]:
#Improved model evaluation
print("Linear Regression Selected Features")
print(sqrt(mean_squared_error(y_train, y_fitted_linear_selected)), '(Train RMSE)')
print(linear_model.score(X_train_selected, y_train), '(Train R^2 Value)')

print(sqrt(mean_squared_error(y_test, y_pred_linear_selected)), '(Test RMSE)')
print(linear_model.score(X_test_selected, y_test), '(Test R^2 Value)')


print("\nTuned Decision Tree Regressor")
print(sqrt(mean_squared_error(y_train, y_fitted_decision_tree)), '(Train RMSE)')
print(decision_tree.score(X_train_capped, y_train), '(Train R^2 Value)')

print(sqrt(mean_squared_error(y_test, y_pred_decision_tree)), '(Test RMSE)')
print(decision_tree.score(X_test_capped, y_test), '(Test R^2 Value)')



### 1.5 Summarize the Findings

The aim of this task was to predict wine quality using supervised regression models. The dataset was explored using summary statistics and visualisations to understand its structure, distributions, outliers, and correlations. No missing values were found. Outliers were handled using IQR capping to reduce their influence without removing valid wine samples. Power Transformation was applied to selected skewed features, followed by standardisation.

Three regression models were developed and evaluated: Linear Regression, Decision Tree Regressor, and KNN Regressor.

Linear Regression achieved a test RMSE of 0.746 and test R² of 0.282. The OLS results showed that citric acid had a high p-value, suggesting that it did not contribute significantly to the model. Citric acid was removed and the model was rebuilt, but its test RMSE remained approximately 0.746 and its test R² slightly decreased to 0.281. Therefore, feature selection did not improve the Linear Regression model.

KNN achieved the best testing performance, with a test RMSE of 0.714 and test R² of 0.341. However, its training RMSE was noticeably lower than its testing RMSE, suggesting some overfitting and weaker generalisation to unseen data.

GridSearchCV improved the Decision Tree compared to its original model. The tuned Decision Tree achieved a test RMSE of 0.725 and test R² of 0.322. Although its testing performance was slightly weaker than KNN, it had a smaller difference between its training and testing results and was easier to interpret.

Therefore, KNN was the most accurate model based on the testing metrics. However, the tuned Decision Tree was selected as the preferred final model because it provided competitive prediction performance, better generalisation, and greater interpretability. Linear Regression was not selected because it produced weaker testing performance, and removing citric acid did not improve its results.

## 2. Task 2: Unsupervised Learning Using Online Retail Data


### 2.1 Load and Explore the    Data

In [ ]:
# Load online retail data
retail = pd.read_csv('./Data/online_retail.csv')

# Basic dataset overview
print("\nHead")
print(retail.head())

print("\nShape")
print(retail.shape)

print("\nInfo")
retail.info()

print("\nDescribe")
print(retail.describe())

print("\nNull Values")
print(retail.isnull().sum())

#The dataset contains transaction-level records. Each row represents a product purchased within an invoice.
#`CustomerID` has missing values, which must be handled before aggregating transactions to the customer level.

In [ ]:
# Boxplots to identify possible outliers
plt.boxplot(retail['Quantity'].dropna())
plt.title('Boxplot of Quantity')
plt.ylabel('Quantity')
plt.show()

plt.boxplot(retail['UnitPrice'].dropna())
plt.title('Boxplot of UnitPrice')
plt.ylabel('UnitPrice')
plt.show()

#The boxplots show extreme values in both Quantity and UnitPrice. 
#This suggests that the dataset contains possible outliers, returns, cancelled transactions, or unusual high-value purchases that need to be considered during data cleaning.
#Also invalied quantities and Unit Prices such as Negative numbers which need to be removed

In [ ]:
# Check invalid or special transaction records
print("Missing CustomerID:", retail['CustomerID'].isnull().sum())
print("Cancelled invoices:", retail['InvoiceNo'].astype(str).str.startswith('C').sum())
print("Quantity <= 0:", (retail['Quantity'] <= 0).sum())
print("UnitPrice <= 0:", (retail['UnitPrice'] <= 0).sum())
#The dataset contains missing customer IDs, cancelled invoices, negative or zero quantities, and non-positive unit prices.
#These records need to be removed or handled because they do not represent normal customer purchases for clustering.

In [ ]:
# Limit extreme values for clearer visualisation
retail_plot = retail[(retail['Quantity'] > 0) & (retail['UnitPrice'] > 0)]

quantity_limit = retail_plot['Quantity'].quantile(0.99)
price_limit = retail_plot['UnitPrice'].quantile(0.99)

retail_plot = retail_plot[
    (retail_plot['Quantity'] <= quantity_limit) &
    (retail_plot['UnitPrice'] <= price_limit)
]

retail_plot[['Quantity', 'UnitPrice']].hist(bins=30, figsize=(10, 4))
plt.suptitle('Distribution of Quantity and UnitPrice Excluding Extreme Outliers')
plt.show()

#The original distributions were heavily affected by extreme outliers, causing most values to be compressed near zero. 
#Therefore, invalid values and extreme outliers were excluded only for visualisation. 
#After excluding extreme values, Quantity and UnitPrice are still right-skewed, which is common in retail transaction data because most customers buy small quantities of lower-priced items.


### 2.2 Cleanse and Transform the Data

In [ ]:
# Make a copy before cleaning
retail_clean = retail.copy()

print("Original shape:", retail_clean.shape)

# Convert InvoiceDate to datetime
retail_clean['InvoiceDate'] = pd.to_datetime(retail_clean['InvoiceDate'], dayfirst=True)

# Remove rows with missing CustomerID
retail_clean = retail_clean.dropna(subset=['CustomerID'])

# Remove cancelled invoices
retail_clean = retail_clean[~retail_clean['InvoiceNo'].apply(lambda x: x[0] == 'C')]

# Remove returns and invalid transactions
retail_clean = retail_clean[retail_clean['Quantity'] > 0]
retail_clean = retail_clean[retail_clean['UnitPrice'] > 0]


print("Cleaned shape:", retail_clean.shape)
print("Rows removed:", retail.shape[0] - retail_clean.shape[0])

#Rows with missing CustomerID were removed because customers cannot be grouped without an ID. 
#Cancelled invoices and negative quantities were removed because they represent cancellations or returns rather than normal purchases.
#UnitPrice values less than or equal to zero were also removed because they are not valid sales transactions.

In [ ]:
# Create transaction total
retail_clean['TotalPrice'] = retail_clean['Quantity'] * retail_clean['UnitPrice']

retail_clean.head()

#AverageBasketValue was included as an additional derived feature because it shows the average spending per invoice. 
#Although it is calculated from Monetary and Frequency, it helps distinguish customers who spend a lot overall from customers who spend a lot per purchase.

In [ ]:
# Create customer-level features
frequency = retail_clean.groupby('CustomerID')['InvoiceNo'].nunique()
monetary = retail_clean.groupby('CustomerID')['TotalPrice'].sum()
last_purchase = retail_clean.groupby('CustomerID')['InvoiceDate'].max()

snapshot_date = retail_clean['InvoiceDate'].max()
recency = (snapshot_date - last_purchase).dt.days

rfm = pd.DataFrame()
rfm['CustomerID'] = frequency.index
rfm['Frequency'] = frequency.values
rfm['Monetary'] = monetary.values
rfm['Recency'] = recency.values
rfm['AverageBasketValue'] = rfm['Monetary'] / rfm['Frequency']

print(rfm.head())

#The transaction data was aggregated to customer level because each cluster should represent a customer segment. 
# Frequency counts the number of unique invoices, Monetary measures total spending, and Recency measures the number of days since the customer's last purchase. 
# AverageBasketValue was also created to show the average spending per invoice.


In [ ]:
for col in ['Recency', 'Frequency', 'Monetary', 'AverageBasketValue']:
    plt.figure(figsize=(5,4))
    plt.boxplot(rfm[col])
    plt.title(f'Boxplot of {col}')
    plt.ylabel(col)
    plt.show()

In [ ]:
# Select features for clustering
X = rfm[['Recency', 'Frequency', 'Monetary', 'AverageBasketValue']]

# Apply IQR capping
winsorizer = Winsorizer(
    capping_method='iqr',
    tail='both',
    fold=1.5,
    variables=X.columns.tolist()
)

winsorizer.fit(X)

X_capped = winsorizer.transform(X)

In [ ]:

# Standardise features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_capped)

#The clustering features were standardised because K-means and Hierarchical Clustering are distance-based methods. 
#Scaling prevents features with larger values, such as Monetary, from dominating the clustering process.

### 2.3 Build the Clustering Models

In [ ]:
# Use the Elbow Method to find a suitable number of clusters for K-means
sse = []

# Run K-means for each value of k
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, n_init=20, random_state=1)
    kmeans.fit(X_scaled)
    sse.append(kmeans.inertia_)

# Plot the Elbow graph
plt.plot(range(1, 11), sse, 'b-*')
plt.title('Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('SSE')
plt.show()

In [ ]:
# Build final K-means model using 4 clusters
kmeans = KMeans(n_clusters=4, n_init=20, max_iter=300, random_state=1)

# Train the model and predict cluster labels
kmeans.fit(X_scaled)
y_pred = kmeans.predict(X_scaled)



In [ ]:
# Create linkage matrix for Hierarchical Clustering
Z = shc.linkage(X_scaled, method='average')

# Plot dendrogram
plt.figure(figsize=(12, 12))
plt.title('Visualising the Hierarchical Clustering')
Dendrogram = shc.dendrogram(Z)
plt.show()

In [ ]:
# Build final Hierarchical Clustering model using 2 clusters
ac2 = AgglomerativeClustering(n_clusters=2, linkage='average')

# Predict cluster labels
rfm['Hierarchical_Cluster'] = ac2.fit_predict(X_scaled)

rfm.head()

#The final Hierarchical Clustering model was built using 2 clusters based on the dendrogram.
#Each customer is assigned to one of two broad customer groups.


### 2.4 Evaluate and Compare the Models

In [ ]:
# Visualise K-means clusters using the first three scaled features
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_scaled[:,0], X_scaled[:,1], X_scaled[:,2], c=y_pred)

# Plot cluster centres
ax.scatter(kmeans.cluster_centers_[:, 0], 
           kmeans.cluster_centers_[:, 1], 
           kmeans.cluster_centers_[:, 2], 
           s=100, c='red')

ax.set_xlabel('Recency_scaled')
ax.set_ylabel('Frequency_scaled')
ax.set_zlabel('Monetary_scaled')

plt.show()

#The 3D scatter plot shows the K-means customer clusters using Recency, Frequency, and Monetary. 
#The red points represent the cluster centres. AverageBasketValue was also used in the model, but it is not shown in the 3D plot because only three dimensions can be displayed.

In [ ]:
# Add K-means cluster labels to the RFM table
rfm['Cluster'] = y_pred
rfm.head()

# Select features used for clustering
dat = rfm[['Recency', 'Frequency', 'Monetary', 'AverageBasketValue']]

# Compare each feature across clusters using boxplots
i = 0

for col in dat: 
    print(col) 
    plt.cla()
    plt.boxplot([dat.iloc[:,i][rfm.Cluster==0].tolist(),
                 dat.iloc[:,i][rfm.Cluster==1].tolist(),
                 dat.iloc[:,i][rfm.Cluster==2].tolist(),
                 dat.iloc[:,i][rfm.Cluster==3].tolist()],
                 tick_labels=('Cluster 0','Cluster 1','Cluster 2','Cluster 3'))    
    
    plt.ylabel(col)
    plt.show()
    i = i + 1

#The boxplots show how each K-means cluster differs based on Recency, Frequency, Monetary, and AverageBasketValue.
 #This helps identify which clusters contain recent customers, frequent buyers, high spenders, or low-value customers.

In [ ]:
kmeans_silhouette = silhouette_score(X_scaled, y_pred)

print("K-means Silhouette Score:", kmeans_silhouette)

In [ ]:
# Summarise average feature values for each K-means cluster
kmeans_summary = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary', 'AverageBasketValue']].mean().round(2)

print(kmeans_summary)

# Count number of customers in each K-means cluster
kmeans_count = rfm.groupby('Cluster')['CustomerID'].count()

print(kmeans_count)

#The K-means summary table shows the average customer behaviour in each cluster. 
#These values will be used to interpret and name the customer segments.

In [ ]:
# Evaluate Hierarchical Clustering using silhouette scores
k_range = range(2,11)
silhouette_scores = []

for i in k_range:
    ac_i = AgglomerativeClustering(n_clusters=i, linkage='average')
    silhouette_scores.append(silhouette_score(X_scaled, ac_i.fit_predict(X_scaled)))

plt.bar(k_range, silhouette_scores) 
plt.xlabel('Number of clusters', fontsize=20) 
plt.ylabel('Silhouette Score', fontsize=20)
plt.show()

# Visualise Hierarchical clusters using the first three scaled features
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.scatter(
    X_scaled[:,0],
    X_scaled[:,1],
    X_scaled[:,2],
    c=rfm['Hierarchical_Cluster']
)

ax.set_xlabel('Recency_scaled')
ax.set_ylabel('Frequency_scaled')
ax.set_zlabel('Monetary_scaled')

plt.show()

#The silhouette score is used to evaluate how well-separated the hierarchical clusters are.
#The highest silhouette score occurs at 2 clusters, which supports the 2-cluster choice from the dendrogram.



In [ ]:
# Evaluate final Hierarchical model
hierarchical_silhouette = silhouette_score(
    X_scaled,
    rfm['Hierarchical_Cluster']
)

print("Hierarchical Silhouette Score:", hierarchical_silhouette)

In [ ]:
# Summarise average feature values for each Hierarchical cluster
hierarchical_summary = rfm.groupby('Hierarchical_Cluster')[['Recency', 'Frequency', 'Monetary', 'AverageBasketValue']].mean().round(2)

print(hierarchical_summary)

# Count number of customers in each Hierarchical cluster
hierarchical_count = rfm.groupby('Hierarchical_Cluster')['CustomerID'].count()

print(hierarchical_count)

#The hierarchical summary table shows that the 2-cluster model gives broader customer groups. 
# This is useful for general separation, but it is less useful for detailed business actions.

### 2.5 Summarize and Interpret

In [ ]:
segment_names = {
    0: 'Recent Low-Value',
    1: 'High-Value Loyal',
    2: 'At-Risk Low-Value',
    3: 'High-Basket Occasional'
}

rfm['SegmentName'] = rfm['Cluster'].map(segment_names)

rfm.head()



In [ ]:

final_summary = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary', 'AverageBasketValue']].mean().round(2)

final_summary['NoOfCustomers'] = rfm.groupby('Cluster')['CustomerID'].count()
final_summary['SegmentName'] = final_summary.index.map(segment_names)

final_summary = final_summary[['SegmentName', 'NoOfCustomers', 'Recency', 'Frequency', 'Monetary', 'AverageBasketValue']]

print(final_summary)

In [ ]:
print("K-means Silhouette Score:", kmeans_silhouette)
print("Hierarchical Silhouette Score:", hierarchical_silhouette)

Cluster 0 was named Recent Low-Value. It contains 2,016 customers who purchased relatively recently but have low purchase frequency, total spending, and average basket value.

Cluster 1 was named High-Value Loyal. These customers have the lowest Recency and the highest Frequency and Monetary value, meaning they purchase recently, frequently, and spend the most overall.

Cluster 2 was named At-Risk Low-Value. These customers have the highest Recency and the lowest Frequency and Monetary value, meaning they have not purchased recently and spend relatively little.

Cluster 3 was named High-Basket Occasional. These customers have moderate Recency and Frequency but the highest Average Basket Value, meaning they purchase occasionally but spend more per purchase.

For Hierarchical Clustering, Cluster 0 represents more active and higher-value customers, while Cluster 1 represents less active and lower-value customers.

Hierarchical Clustering achieved a higher Silhouette Score of 0.448 compared to 0.409 for K-means. However, K-means was selected because its four clusters provided more detailed and interpretable customer segments than the two broad Hierarchical clusters.